# B2-021 — Session 4: Graphs and Cross-modal Attention

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260901`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, model hubs, pretrained models, checkpoints, or student data.


## 1. Graphs, nodes, edges, and adjacency

Each graph has four nodes. Node features are CPU float32 `(B,4,3)` and adjacency is uint8 `(B,4,4)`, where `A[b,i,j]=1` means node `i` receives a message from node `j`. The fixture includes self-loops. Reject nonbinary entries, asymmetry when an undirected graph is promised, or edges to nonexistent nodes.

**Checkpoint 1A.** Which adjacency axis names the receiver?

**Checkpoint 1B.** What is the node-feature axis?

**Collected answers.** Axis 1 of adjacency is receiver `i`; axis 1 of node features indexes nodes.


## 2. Neighbor aggregation with explicit axes

Convert adjacency to float32. Degree is `A.sum(dim=-1,keepdim=True)` shape `(B,4,1)`. Mean messages are `A @ X / degree.clamp_min(1)`, shape `(B,4,3)`, because the last adjacency axis contracts with the node axis of `X`. A learned `Linear(3,8)` then makes graph tokens `(B,4,8)`.

**Checkpoint 2A.** Which axis is summed for degree?

**Checkpoint 2B.** Why keep a singleton feature axis?

**Collected answers.** Sum sender axis `j`; `(B,4,1)` broadcasts cleanly over three features.


## 3. Permutation equivariance before positions

Let permutation matrix `P` reorder nodes. Reordered features and adjacency are `PX` and `PAP^T`. Sum aggregation becomes `(PAP^T)(PX)=P(AX)` because `P^T P = I`. Mean aggregation divides by receiver degree, `D^{-1}AX`; relabeling permutes the degrees too (`PDP^T`), so `(PDP^T)^{-1}(PAP^T)(PX)=P(D^{-1}AX)`. In both cases output rows permute as `P`. A graph-level mean or invariant class token then removes row order. Arbitrary absolute node positions can break this property.

**Checkpoint 3A.** What changes under node relabeling?

**Checkpoint 3B.** What graph readout is invariant?

**Collected answers.** Node outputs reorder but values correspond; mean over nodes is invariant.


## 4. Graph tokens and a classifier

After mean aggregation and `Linear(3,8)`, prepend one learned graph-class token to get `(B,5,8)`. One-head self-attention lets every graph token communicate, while the class-token output `(B,8)` feeds `Linear(8,2)`. This mirrors ViT token flow but adjacency aggregation supplies explicit relational structure first.

**Checkpoint 4A.** How many tokens enter attention?

**Checkpoint 4B.** Where does adjacency act?

**Collected answers.** Five; adjacency acts before token projection/attention in this tiny model.


## 5. Cross-modal attention ownership

If image tokens ask which graph facts matter, image states own `Q` and graph states own `K,V`; output count equals image-query count. Reversing the task—updating graph nodes from image evidence—reverses ownership. The score shape `(B,N_image,N_graph)` records the direction. Keys select compatibility; values carry retrieved content.

**Checkpoint 5A.** Who owns values when images read a graph?

**Checkpoint 5B.** What determines output sequence length?

**Collected answers.** Graph states own values; image queries determine output length.


## 6. Fixed-seed tiny graph-token train/eval trace

This recorded reference-run lesson-only mini-trace uses three-node graphs with identity adjacency, two-dimensional node features, four training graphs, and two distinct held-out graphs. A mean-aggregate/linear classifier uses seed `20260901` and 12 AdamW updates at `lr=.08`; its held-out aggregate shape is `(2,3,2)`. It does not import the four-node p20 fixture.

| update | train CE | held-out CE | held-out accuracy |
|---:|---:|---:|---:|
| 0 | 0.851557 | 0.851395 | 0.00 |
| 1 | 0.722031 | 0.721996 | 0.50 |
| 6 | 0.286516 | 0.285071 | 1.00 |
| 12 | 0.098204 | 0.095487 | 1.00 |

The aggregate shape proves the node axis was preserved before graph-level mean readout. Only the four training graphs enter backward; the two held-out graphs are evaluated without updates. p20 adds a class token and attention at different width/lr, so these values teach trace interpretation without revealing its answer.

**Checkpoint 6A.** Can labels be concatenated to node features?

**Checkpoint 6B.** How do the lesson demonstration and p20 update-count/lr protocols differ?

**Collected answers.** No, labels in features leak targets. The lesson-only demonstration uses exactly 12 updates at `lr=.08`; p20 later requires exactly 12 updates at `lr=.05`.


## 7. Worked laboratory: directed messages and modal ownership

Read adjacency as a sentence: `A[i,j]=1` means receiver node `i` reads sender node `j`. This convention permits directed graphs. A self-loop lets a node retain its own feature. Degree sums the sender axis `j`, so its shape is `(B,N,1)`. Batched multiplication `A@X` then forms one weighted sender sum for every receiver and feature. Dividing by receiver degree produces mean messages without looping over nodes.

The first code cell uses a directed asymmetric adjacency. Receiver zero reads itself and node one, while receiver one does not read node zero. Consequently `A@X` differs from `A.T@X`. A symmetric chain would be a weak witness because transposition leaves it unchanged. Compare the complete expected aggregate, not just receiver zero; an axis-swap bug may accidentally preserve one row.

After aggregation, a shared `Linear(3,8)` acts row-wise. Prepend one learned graph-class token to four projected node tokens. Self-attention uses this five-token tensor as Q, K, and V; add its output residually, normalize, then classify normalized token zero. The class token supplies an invariant graph-level readout only if no absolute feature silently identifies a node order. Relabeling nodes with permutation `P` requires transforming both `X` and adjacency as `PX` and `PAP^T`.

Cross-modal attention has a different ownership question. If image tokens are updated from graph facts, image tokens are queries and graph tokens are keys and values. Score shape is `(B,N_image,N_graph)`, and output count is `N_image`. Values follow keys because each compatibility score weights a graph value. Reversing Q with K/V changes which modality is updated even if the matrix multiplication still has legal dimensions.

The second code cell writes a tiny Q/K/V shape trace. It intentionally uses unequal token counts so a reversal cannot hide behind a square score matrix. Before running, predict both score and output shapes. Then state the semantic consequence: every image query receives a weighted combination of graph values. Masks must remove padded graph keys and values; an edge touching a padded node is invalid before attention begins.

Data leakage can enter graph systems through structure, not only labels. An edge constructed from a target, a held-out graph included in optimizer IDs, or nodes from one graph split across train and held-out all leak information. Actual structured-feature fingerprints must include both node features and adjacency in fixed field order. A repeated class label is harmless; a repeated node-plus-adjacency fingerprint is not.

For p20, record baseline and twelve updates at the fixed learning rate. Do not replace adjacency aggregation with attention: they represent separate stages and the aggregate probe must have shape `(B,4,3)`. If performance fails, inspect directed aggregate bytes first, token construction second, and residual/norm order third. Increasing steps would violate the certificate rather than diagnose it.

As a final hand calculation, choose receiver two in the first practice graph and list exactly which sender rows contribute. Add those feature rows, divide by that receiver's degree, and compare all three coordinates with the executable tensor. Then relabel nodes zero and three: apply the same permutation to node rows and to both adjacency axes. The aggregate must permute in the same way. If only adjacency rows move, sender identity is corrupted; if only columns move, receiver identity is corrupted. This small exercise connects the matrix proof to concrete bytes and makes “permutation equivariant” a testable claim rather than a slogan.

**Checkpoint 7A.** How does the asymmetric adjacency witness distinguish `A@X` from `A.T@X`?

**Checkpoint 7B.** When image tokens read graph facts, who owns Q, K, and V and what is the score shape?

**Collected answers.** A directed receiver/sender pattern changes under transpose, so the complete aggregates differ; image owns Q, graph owns K and V, and scores have shape `(B,N_image,N_graph)`.


In [ ]:
import torch

x = torch.tensor([[[1.,0.],[0.,2.],[3.,1.]]])
a = torch.tensor([[[1,1,0],[0,1,1],[1,0,1]]], dtype=torch.uint8)
degree = a.sum(-1, keepdim=True)
mean = torch.bmm(a.float(), x) / degree
wrong = torch.bmm(a.transpose(1,2).float(), x) / a.transpose(1,2).sum(-1,keepdim=True)
assert not torch.allclose(mean, wrong, atol=1e-6, rtol=1e-6)
print('A@X mean:', mean)
print('A.T@X mean:', wrong)


In [ ]:
import torch

image_q = torch.randn(2, 5, 8)
graph_kv = torch.randn(2, 3, 8)
scores = image_q @ graph_kv.transpose(-2, -1) / (8 ** .5)
weights = scores.softmax(-1)
updated_images = weights @ graph_kv
assert scores.shape == (2,5,3) and updated_images.shape == (2,5,8)
print(tuple(scores.shape), tuple(updated_images.shape))


## 8. Common pitfalls and Exam connections

Common pitfalls are transposing sender/receiver meaning without updating the formula, dividing along the feature axis, omitting isolated-node protection, adding target-derived edges, splitting nodes from one graph across train and held-out, and reversing cross-modal Q versus K/V. Exam connections present a tiny relabeling or trace and ask which invariant breaks first.

**Checkpoint 8A.** Which practice implements axes?

**Checkpoint 8B.** Which practice audits Q/K/V reversal?

**Collected answers.** p12 implements aggregation; p24 audits cross-modal ownership.


## 9. Forward-only deeper topic: sparse and geometric graph Transformers

Forward-only deeper topic: large systems use sparse edge attention, graph positional encodings, equivariant geometric features, and heterogeneous node types. They are not assessed here. B2-021 keeps four literal nodes, dense adjacency, explicit mean aggregation, and ordinary attention so permutation and leakage claims can be checked exactly.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Are Laplacian positions required?

**Checkpoint 9B.** What graph size is executable?

**Collected answers.** No; four nodes per graph.
